In [2]:
import os
from dotenv import load_dotenv
import duckdb

load_dotenv(override=True)

con = duckdb.connect()

In [3]:
path = "s3://mateom/graal/embeddings/NAF2025/2026-03-16_gemma_synth.parquet"

In [4]:
code_len_query = f"""
    SELECT COUNT(DISTINCT(code))
    FROM read_parquet('{path}')
    """

n_codes = con.execute(code_len_query).fetchone()[0]

In [5]:
n_codes

747

In [6]:
random_state = 42

In [7]:
counts = con.execute(f"""
    SELECT code, COUNT(*) AS cnt
    FROM read_parquet('{path}')
    GROUP BY code
    ORDER BY code
""").df()

In [8]:
import numpy as np

rng = np.random.default_rng(seed=random_state)

counts["k"] = counts["cnt"].apply(
    lambda n: rng.integers(1, n + 1)
)

In [9]:
counts["offset"] = counts["cnt"].cumsum() - counts["cnt"]
counts["row_id"] = counts["offset"] + counts["k"]

In [10]:
selected_ids = counts[["row_id"]]
con.register("selected_ids", selected_ids)

In [11]:
selected_ids = counts["row_id"].to_numpy()
remaining_ids = np.delete(np.arange(1,counts["cnt"].sum()+1),selected_ids-1)
random_ids = rng.choice(remaining_ids, 300)
final_ids = np.concatenate([selected_ids, random_ids])
con.register("selected_ids", {"row_id":final_ids})

In [12]:
query = f"""
SELECT *
FROM selected_ids
"""

res = con.execute(query).df()

In [13]:
query = f"""
SELECT t.code, t.embedding
FROM (
    SELECT *,
           ROW_NUMBER() OVER (ORDER BY code) AS row_id
    FROM read_parquet('{path}')
) t
JOIN selected_ids s
USING (row_id)
"""

res = con.execute(query).df()

In [14]:
res.head()

,code,embedding
0,96.30Y,"[0.039255790412425995, 0.01053204108029604, -0..."
1,96.40Y,"[0.029340434819459915, 0.02590949647128582, 0...."
2,96.91Y,"[0.015917960554361343, 0.031123176217079163, 0..."
3,96.99G,"[-0.0019990347791463137, 0.012319977395236492,..."
4,96.99H,"[0.005781588610261679, 0.01179560273885727, 0...."


In [15]:
code_list = res["code"].to_list()

In [16]:
from collections import Counter

code_counts = Counter(code_list)

In [17]:
code_counts["01.11Y"]

5

In [50]:
code_row_count = con.execute(f"""
    SELECT code, COUNT(*) AS n_rows
    FROM read_parquet('{path}')
    GROUP BY code
    ORDER BY code
    """).df()

In [51]:
code_row_count["row_id"] = code_row_count.apply(
    lambda x: rng.integers(1, x["n_rows"] + 1, size=code_counts[x["code"]]), axis=1
)

In [52]:
code_row_count["offset"] = code_row_count["n_rows"].cumsum() - code_row_count["n_rows"]
code_row_count["row_id"] = code_row_count["offset"] + code_row_count["row_id"]

In [53]:
all_row_ids = code_row_count["row_id"].explode().to_frame()

In [55]:
code_row_count = code_row_count.drop(columns=["row_id"]).join(all_row_ids, how="right", lsuffix="_list")

In [56]:
con.register("selected_ids", code_row_count[["row_id"]])

query = f"""
SELECT t.code, t.embedding
FROM (
    SELECT *,
        ROW_NUMBER() OVER (ORDER BY code) AS row_id
    FROM read_parquet('{path}')
) t
JOIN selected_ids s
USING (row_id)
"""

sampled_df = con.execute(query).df()

In [57]:
sampled_df

,code,embedding
0,96.30Y,"[0.03801770508289337, 0.016157524660229683, 0...."
1,96.40Y,"[0.03641705587506294, 0.025353645905852318, 0...."
2,96.91Y,"[0.03334799036383629, 0.012121167033910751, -0..."
3,96.99G,"[-0.013255584985017776, 0.01842704601585865, -..."
4,96.99H,"[0.03894428908824921, 0.010042238049209118, -0..."
...,...,...
1042,96.21G,"[0.02038068138062954, 0.02084920182824135, 0.0..."
1043,96.21G,"[0.02798541821539402, 0.014348456636071205, 0...."
1044,96.21H,"[0.026793481782078743, 0.02832118049263954, 0...."
1045,96.22Y,"[0.018229331821203232, 0.023561721667647362, 0..."
